In [ ]:
# Before running, install these packages:
# pip install torchcodec opencv-python matplotlib accelerate

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_path = "Alibaba-DAMO-Academy/RynnBrain-CoP-8B"
cache_dir = r"E:\Master - Polimi\Thesis\Practical steps\RynnBrain\hf_cache"

video_path = r"E:\Master - Polimi\Thesis\Practical steps\data\videos\your_video.mp4"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    dtype = torch.float16
else:
    dtype = torch.float32

processor = AutoProcessor.from_pretrained(
    model_path,
    cache_dir=cache_dir,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

model.eval()

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": video_path},
            {
                "type": "text",
                "text": (
                    "This video shows a robotic manipulation task. "
                    "Analyze the video using physically grounded reasoning. "
                    "Focus on the robot motion, the manipulated object, contact/grasp quality, "
                    "and the final state. "
                    "Explain whether the task appears successful or failed."
                )
            },
        ],
    }
]

model_inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

model_inputs = model_inputs.to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False
    )

output_ids = output_ids[:, model_inputs["input_ids"].shape[1]:]

response = processor.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(response)